In [1]:
import pandas as pd

path = "../data/ecom-user-churn-data.csv"
df = pd.read_csv(path)

df.shape, df.columns[:10]

((49358, 49),
 Index(['visitorid', 'ses_rec', 'ses_rec_avg', 'ses_rec_sd', 'ses_rec_cv',
        'user_rec', 'ses_n', 'ses_n_r', 'int_n', 'int_n_r'],
       dtype='str'))

In [ ]:
# Basic structure
print("Rows, Cols:", df.shape)
display(df.head())

# Target distribution (churn rate)
churn_dist = df["target_class"].value_counts(dropna=False)
churn_rate = df["target_class"].value_counts(normalize=True)

print("\nTarget distribution:\n", churn_dist)
print("\nTarget %:\n", (churn_rate * 100).round(2))

# Revenue sanity check
print("\nRevenue summary (rev_sum):")
display(df["rev_sum"].describe())

Rows, Cols: (49358, 49)


,visitorid,ses_rec,ses_rec_avg,ses_rec_sd,ses_rec_cv,user_rec,ses_n,ses_n_r,int_n,int_n_r,...,int_cat16_n,int_cat17_n,int_cat18_n,int_cat19_n,int_cat20_n,int_cat21_n,int_cat22_n,int_cat23_n,int_cat24_n,target_class
0,89,23.0,16.0,0.0,0.0,39.0,2.0,0.051282,3.0,1.500000,...,0,0,0,0,3,0,0,0,0,1
1,251,23.0,61.0,0.0,0.0,85.0,2.0,0.023529,2.0,1.000000,...,0,0,0,0,0,0,1,0,0,1
2,270,14.0,0.1,0.3,3.0,21.0,11.0,0.523810,13.0,1.181818,...,0,0,0,0,0,0,13,0,0,1
3,298,11.0,13.0,0.0,0.0,24.0,2.0,0.083333,4.0,2.000000,...,0,0,0,3,1,0,0,0,0,1
4,474,16.0,26.0,0.0,0.0,43.0,2.0,0.046512,2.0,1.000000,...,2,0,0,0,0,0,0,0,0,1



Target distribution:
 target_class
1    43711
0     5647
Name: count, dtype: int64

Target %:
 target_class
1    88.56
0    11.44
Name: proportion, dtype: float64

Revenue summary (rev_sum):


count    4.935800e+04
mean     1.197114e+04
std      3.475669e+05
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      5.971796e+07
Name: rev_sum, dtype: float64

In [ ]:
active = df[df["target_class"] == 1]
churned = df[df["target_class"] == 0]

print("Avg revenue - Active:", active["rev_sum"].mean())
print("Avg revenue - Churned:", churned["rev_sum"].mean())

print("\nTotal revenue - Active:", active["rev_sum"].sum())
print("Total revenue - Churned:", churned["rev_sum"].sum())

print("\n% Revenue from churned users:",
      churned["rev_sum"].sum() / df["rev_sum"].sum() * 100)

Avg revenue - Active: 8710.458786117912
Avg revenue - Churned: 37210.70798654153

Total revenue - Active: 380742864.0
Total revenue - Churned: 210128868.0

% Revenue from churned users: 35.56251833012042


In [ ]:
df["rev_tier"] = pd.qcut(
    df["rev_sum"].rank(method="first"),
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

churn_by_tier = df.groupby("rev_tier")["target_class"].value_counts(normalize=True).unstack()

churn_by_tier

target_class,0,1
rev_tier,,
Low,0.111183,0.888817
Medium,0.109328,0.890672
High,0.118000,0.882000
Very High,0.119125,0.880875


In [ ]:
# Create high value flag (top 25%)
threshold = df["rev_sum"].quantile(0.75)

df["high_value"] = df["rev_sum"] >= threshold

# Revenue breakdown
segment_summary = df.groupby(["high_value", "target_class"])["rev_sum"].agg(["count", "sum"])

segment_summary

count          sum
high_value target_class                    
True       0              5647  210128868.0
           1             43711  380742864.0

In [12]:
# Revenue-paying vs non-paying users
df["is_payer"] = df["rev_sum"] > 0

segment_summary = df.groupby(["is_payer", "target_class"])["rev_sum"].agg(["count", "sum"])

segment_summary

count          sum
is_payer target_class                    
False    0              5303          0.0
         1             42067          0.0
True     0               344  210128868.0
         1              1644  380742864.0

In [13]:
# Subset only paying users
payers = df[df["is_payer"] == True]

# Compare key behavioural metrics
metrics = [
    "ses_rec",
    "ses_n",
    "int_n",
    "tran_n",
    "ses_len_avg",
    "time_to_tran",
    "ses_n_r"
]

comparison = payers.groupby("target_class")[metrics].mean()

comparison

,ses_rec,ses_n,int_n,tran_n,ses_len_avg,time_to_tran,ses_n_r
target_class,,,,,,,
0,9.610465,20.918605,88.188953,5.601744,6.596855,37.117076,0.616699
1,16.074818,5.824209,24.816910,2.167275,6.432240,18.863945,0.250836


In [14]:
# Compare distribution of time_to_tran
payers.groupby("target_class")["time_to_tran"].describe()

,count,mean,std,min,25%,50%,75%,max
target_class,,,,,,,,
0,344.0,37.117076,82.650327,0.998148,10.004166,19.766667,39.120835,1233.33340
1,1644.0,18.863945,23.630319,0.000000,5.812500,12.420834,23.195833,282.10556


In [15]:
payers["gap_risk"] = payers["time_to_tran"] > 30

risk_summary = payers.groupby("gap_risk")["target_class"].value_counts(normalize=True).unstack()

risk_summary

target_class,0,1
gap_risk,,
False,0.144110,0.855890
True,0.290816,0.709184


In [16]:
# Revenue at risk (gap_risk = True)
rev_at_risk = payers[payers["gap_risk"] == True]["rev_sum"].sum()

total_revenue = df["rev_sum"].sum()

print("Revenue at risk:", rev_at_risk)
print("Percent of total revenue at risk:", rev_at_risk / total_revenue * 100)

Revenue at risk: 221162508.0
Percent of total revenue at risk: 37.42986777373875


In [17]:
# Calculate expected revenue loss from gap-risk group
gap_users = payers[payers["gap_risk"] == True]

expected_loss = gap_users["rev_sum"].sum() * 0.290816

print("Expected revenue loss from gap-risk group:", expected_loss)

Expected revenue loss from gap-risk group: 64317595.92652801


In [18]:
reduced_churn = 0.20

expected_loss_reduced = gap_users["rev_sum"].sum() * reduced_churn

revenue_saved = expected_loss - expected_loss_reduced

print("Potential revenue saved:", revenue_saved)

Potential revenue saved: 20085094.326528005


In [19]:
bins = [0, 15, 30, 45, 1000]
labels = ["0-15", "15-30", "30-45", "45+"]

payers["gap_bucket"] = pd.cut(payers["time_to_tran"], bins=bins, labels=labels)

bucket_summary = payers.groupby("gap_bucket")["target_class"].value_counts(normalize=True).unstack()

bucket_summary

target_class,0,1
gap_bucket,,
0-15,0.125581,0.874419
15-30,0.191147,0.808853
30-45,0.238579,0.761421
45+,0.340206,0.659794


In [20]:
rev_45plus = payers[payers["gap_bucket"] == "45+"]["rev_sum"].sum()

total_revenue = df["rev_sum"].sum()

print("Revenue in 45+ bucket:", rev_45plus)
print("Percent of total revenue:", rev_45plus / total_revenue * 100)

Revenue in 45+ bucket: 62154360.0
Percent of total revenue: 10.519095200174512


In [21]:
export_cols = [
    "visitorid",
    "rev_sum",
    "target_class",
    "gap_bucket",
    "time_to_tran",
    "ses_n",
    "tran_n",
    "int_n"
]

df_export = df.merge(
    payers[["visitorid", "gap_bucket"]],
    on="visitorid",
    how="left"
)

df_export.to_csv("../outputs/churn_bi_table.csv", index=False)